# Samriddha Raj Satyal. Student ID: 240488

### Importing Necessary Pyspark Libraries for data preprocessing and creating spark session

In [1]:
from pyspark.sql import SparkSession,functions as F
from pyspark.sql.types import StructType, StructField,StringType,IntegerType,FloatType

In [2]:
spark=SparkSession.builder.appName('Data Analysis').master('local[*]').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 10:51:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 10:51:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Part A — Loading & Inspection
## Q1. `` Load students_data.csv into a DataFrame using inferSchema=True. Print the schema and display the first 5 rows.

## Q2. `` How many rows and columns does the dataset have? Print both counts.

## Q3. `` Redefine the schema manually using StructType. Set gpa as FloatType and student_id as IntegerType, then reload the CSV with this schema.

In [3]:
students_df=spark.read.csv('students_data-d718fb6b-e9ae-4c0a-97cb-d45b46a66e15.csv',inferSchema=True,header=True)

#Question 1:
students_df.printSchema()
students_df.show(5)

#Question 2:
print('Rows:',students_df.count())
print('Columns:',len(students_df.columns))



# Question 3:
manual_schema = StructType([
    StructField('student_id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('gender', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('department', StringType(), True),
    StructField('year', IntegerType(), True),
    StructField('gpa', FloatType(), True),
    StructField('scholarship', StringType(), True), 
    StructField('city', StringType(), True),
    StructField('email', StringType(), True)
])
students_df = spark.read.csv('students_data-d718fb6b-e9ae-4c0a-97cb-d45b46a66e15.csv', header=True, schema=manual_schema)
students_df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- gpa: double (nullable = true)
 |-- scholarship: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)

+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|student_id|         name|gender|age|      department|year| gpa|scholarship|     city|               email|
+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|       101| Aarav Sharma|  Male| 20|Computer Science|   2| 3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       102|  Priya Thapa|Female| 22|        Business|   4| 3.2|         No|  Pokhara| priya.thapa@uni.edu|
|       103|  Rohan Karki|  Male| 21|     Engineering|   3|NULL|       

# Part B — Null Value Handling
## Q4. `` Count the number of null values in each column. Which columns have missing data?

## Q5. `` Filter and display only the rows that have a null value in the gpa column.

## Q6. `` Fill null values in gpa with the mean GPA of all non-null students. Fill null department with "Undeclared" and null age with 20.

In [4]:
#Question 4:
null_counts=students_df.select(
    [
        F.sum(
            F.when(
                F.col(c).isNull(),1
            ).otherwise(0)
        ).alias(c)
        for c in students_df.columns
    ] 
)

null_counts.show()


#Question 5
students_df.filter(
    F.col('gpa').isNull()
).show()

#Question 6
students_df_filled=students_df.fillna({
    'gpa':students_df.select(F.mean('gpa')).collect()[0][0],
    'department':'Undeclared',
    'age':20
})

students_df_filled.show()

+----------+----+------+---+----------+----+---+-----------+----+-----+
|student_id|name|gender|age|department|year|gpa|scholarship|city|email|
+----------+----+------+---+----------+----+---+-----------+----+-----+
|         0|   0|     0|  2|         1|   0|  3|          0|   0|    0|
+----------+----+------+---+----------+----+---+-----------+----+-----+

+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|student_id|          name|gender|age| department|year| gpa|scholarship|    city|               email|
+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|       103|   Rohan Karki|  Male| 21|Engineering|   3|NULL|         No|Lalitpur| rohan.karki@uni.edu|
|       110|  Nisha Tamang|Female| 20|       Arts|   2|NULL|         No|Lalitpur|nisha.tamang@uni.edu|
|       122|Dipika Niroula|Female| 20|   Business|   2|NULL|         No|  Dharan|dipika.niroula@un...|
+----------+---------

# Part C — Selecting & Filtering
## Q7. `` Select only name, department, and gpa. Display the result.

## Q8. `` Show all students from the Computer Science department who have a GPA above 3.5.

## Q9. `` Display students who are in year 3 or year 4 AND have a scholarship. Use isin() for the year condition.

## Q10. `` Add a new column grade: assign "Distinction" if GPA ≥ 3.7, "Merit" if GPA ≥ 3.0, else "Pass".

In [11]:
#Question 7:
students_df.select('name','department','gpa').show()

#Question 8:
students_df.filter((F.col('department')=='Computer Science') & (F.col('gpa')>3.5)).show()

#Question 9:
students_df.filter((F.col('year').isin(3,4)) & (F.col('scholarship')=='Yes')).show()

#Question 10:
students_df_added=students_df.withColumn('grade',
                                        F.when(F.col('gpa')>=3.7,'Distinction')
                                         .when(F.col('gpa')>=3.0,'Merit')
                                         .otherwise('Pass')
                                        )

students_df_added.show()

+----------------+----------------+----+
|            name|      department| gpa|
+----------------+----------------+----+
|    Aarav Sharma|Computer Science| 3.8|
|     Priya Thapa|        Business| 3.2|
|     Rohan Karki|     Engineering|NULL|
|        Sita Rai|Computer Science| 3.9|
|   Bikash Gurung|        Business| 2.7|
|  Anita Shrestha|     Engineering| 3.5|
|   Deepak Pandey|            Arts| 3.1|
|     Kavya Joshi|Computer Science| 3.7|
|    Suresh Limbu|     Engineering| 2.9|
|    Nisha Tamang|            Arts|NULL|
|   Prakash Bista|        Business| 3.4|
|     Manisha Oli|Computer Science| 3.6|
| Rajesh Adhikari|     Engineering| 3.0|
|    Sunita Magar|            Arts| 2.8|
|    Kiran Poudel|Computer Science| 3.3|
|     Rupa Basnet|            NULL| 3.7|
|  Anil Chaudhary|     Engineering| 2.6|
|    Puja Koirala|        Business| 3.5|
|     Nabin Dahal|            Arts| 3.2|
|Kabita Bhattarai|Computer Science| 3.9|
+----------------+----------------+----+
only showing top

# Part D — Dropping & Renaming
## Q11. `` Drop the email and student_id columns from the DataFrame. Confirm by printing the column names.

## Q12. `` Remove all rows where department is null, then remove all duplicate rows based on name and department together.

## Q13. `` Rename gpa to grade_point_avg and year to study_year using withColumnRenamed.

## Q14. `` Rename all columns at once by replacing underscores with spaces using toDF(). Print the new column names.

In [16]:
#Question 11:
students_df_dropped=students_df.drop('email','student_id')
print(students_df_dropped.columns)

#Question 12:
students_df_removed_null_rows=students_df.dropna(subset=['department']).dropDuplicates(['name','department'])
students_df_removed_null_rows.show()


#Question 13:
students_renamed = students_df.withColumnRenamed('gpa', 'grade_point_avg')\
                              .withColumnRenamed('year', 'study_year')

students_renamed.show()

#Question 14:
new_column_names = [c.replace('_', ' ') for c in students_df.columns]

students_spaced_cols = students_df.toDF(*new_column_names)
print("New bulk names:", students_spaced_cols.columns)

['name', 'gender', 'age', 'department', 'year', 'gpa', 'scholarship', 'city']
+----------+----------------+------+----+----------------+----+----+-----------+---------+--------------------+
|student_id|            name|gender| age|      department|year| gpa|scholarship|     city|               email|
+----------+----------------+------+----+----------------+----+----+-----------+---------+--------------------+
|       101|    Aarav Sharma|  Male|  20|Computer Science|   2| 3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       117|  Anil Chaudhary|  Male|  21|     Engineering|   3| 2.6|         No|Nepalgunj|anil.chaudhary@un...|
|       106|  Anita Shrestha|Female|  20|     Engineering|   2| 3.5|        Yes|Bhaktapur|anita.shrestha@un...|
|       105|   Bikash Gurung|  Male|  23|        Business|   4| 2.7|         No|   Butwal|bikash.gurung@uni...|
|       125|  Bishal Ghimire|  Male|  23|     Engineering|   4| 2.8|         No|Kathmandu|bishal.ghimire@un...|
|       107|   Deepak Pand

# Part E — Statistics & Aggregation
## Q15. `` Run describe() on gpa and age. What is the mean GPA? What is the minimum age?

## Q16. `` Find the average GPA per department. Sort the result from highest to lowest average GPA.

## Q17. `` Count how many students are on scholarship vs not, broken down by gender. (Hint: group by two columns.)

## Q18. `` For each city, find the number of students, average GPA, and count of scholarship holders. Display cities with more than 2 students only.

In [20]:
#Question 15:
students_df.describe('gpa', 'age').show()

#Question 16:
students_df.groupby('department') \
           .agg(F.mean('gpa').alias('avg_gpa')) \
           .orderBy(F.col('avg_gpa').desc()) \
           .show()


#Questioon 17:
students_df.groupby('gender', 'scholarship').count().show()

#Question 18:
students_df.groupby('city') \
           .agg(
               F.count('student_id').alias('student_count'),
               F.mean('gpa').alias('avg_gpa'),
               F.sum(F.when(F.col('scholarship') == 'Yes', 1).otherwise(0)).alias('scholarship_count')
           ) \
           .filter(F.col('student_count') > 2) \
           .show()

+-------+-------------------+-----------------+
|summary|                gpa|              age|
+-------+-------------------+-----------------+
|  count|                 22|               23|
|   mean|  3.304545456712896|21.08695652173913|
| stddev|0.39457908784751333|1.411415906690894|
|    min|                2.6|               19|
|    max|                3.9|               24|
+-------+-------------------+-----------------+

+----------------+-----------------+
|      department|          avg_gpa|
+----------------+-----------------+
|            NULL|3.700000047683716|
|Computer Science|3.614285707473755|
|        Business|3.200000047683716|
|            Arts|3.174999952316284|
|     Engineering|3.033333341280619|
+----------------+-----------------+

+------+-----------+-----+
|gender|scholarship|count|
+------+-----------+-----+
|  Male|         No|   10|
|  Male|        Yes|    3|
|Female|         No|    5|
|Female|        Yes|    7|
+------+-----------+-----+

+---------+-----